# EMiF Project — Risk Structure After COVID-19

## Research question

Can we say that, since COVID-19, the structure of risk in financial markets has changed?

This notebook reproduces the empirical analysis used in the report. The analysis uses only the dataset provided in `Data.xlsx` and does not require external market data or hidden Python modules.

We define the "structure of risk" as five dimensions:

1. Risk levels: volatility, tail risk, skewness and kurtosis.
2. Risk co-movement: pairwise correlations and asset-class block correlations.
3. Full co-movement structure: the global correlation-matrix shift.
4. Risk factors: principal-component concentration and common shocks.
5. Portfolio implications: diversification, minimum-variance weights and equal-risk-contribution weights.

The notebook combines descriptive evidence with formal bootstrap tests, Fisher correlation tests, a correlation-matrix distance test, a structural-break analysis of equity-bond diversification, and portfolio exercises.


In [ ]:
from pathlib import Path
import math
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

try:
    from scipy.optimize import minimize
except ImportError:
    minimize = None

print("Libraries imported successfully.")


In [ ]:
CURRENT_DIR = Path.cwd()

possible_paths = [
    CURRENT_DIR / "Data.xlsx",
    CURRENT_DIR / "data" / "Data.xlsx",
    CURRENT_DIR / "Data (2).xlsx",
    CURRENT_DIR.parent / "Data.xlsx",
    CURRENT_DIR.parent / "data" / "Data.xlsx",
]

DATA_PATH = None

for path in possible_paths:
    if path.exists():
        DATA_PATH = path
        break

if DATA_PATH is None:
    raise FileNotFoundError(
        "Data.xlsx not found. Please place Data.xlsx in the same folder as the notebook "
        "or in a data/ folder."
    )

print("Using data file:", DATA_PATH)
print("Notebook mode: self-contained analysis code with inline outputs.")
print("No tables or figures are written to a results folder.")


## Helper functions

The following helper functions are included directly in the notebook to make the analysis fully reproducible from the notebook and `Data.xlsx`, without requiring external Python modules.

In [ ]:
def load_raw_data(path):
    """
    Load the raw Excel file provided for the EMiF project.
    The function assumes the first column is a date column.
    """
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")

    df = pd.read_excel(path)

    df.columns = (
        df.columns.astype(str)
        .str.strip()
        .str.replace("\n", " ", regex=False)
        .str.replace("  ", " ", regex=False)
    )

    date_col = df.columns[0]
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce")

    df = df.dropna(subset=[date_col])
    df = df.set_index(date_col)
    df.index.name = "Date"
    df = df.sort_index()

    return df

In [ ]:
LOG_RETURN_COLUMNS = [
    "S&P500",
    "Eurostoxx 50",
    "Hang Seng",
    "MSCI EM",
    "SMI",
    "Gold",
    "EURUSD",
    "USDJPY",
    "US IG Bonds",
    "US HY Bonds",
    "USDCHF",
]

YIELD_COLUMNS = [
    "US T 10-year Yield",
    "German Gov 10-year yield",
]


def prepare_returns(raw_df):
    """
    Convert raw market data into stationary financial series.

    - Price/index/bond/FX series are transformed into log returns.
    - Government bond yields are transformed into daily changes in basis points.
    - Oil futures are transformed with an asinh difference because oil prices became
      negative in April 2020, making log returns impossible.
    """

    df = raw_df.copy()
    df.index.name = "Date"

    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    transformed = pd.DataFrame(index=df.index)

    for col in LOG_RETURN_COLUMNS:
        transformed[col] = np.log(df[col] / df[col].shift(1))

    for col in YIELD_COLUMNS:
        transformed[col] = df[col].diff() * 100

    transformed["Oil futures"] = (
        np.arcsinh(df["Oil futures"]) - np.arcsinh(df["Oil futures"].shift(1))
    )

    transformed = transformed[df.columns]
    transformed = transformed.dropna(how="any")

    return transformed


def split_pre_post_covid(returns_df, breakpoint="2020-03-11"):
    """
    Split the dataset into pre-COVID and post-COVID samples.
    """

    breakpoint = pd.to_datetime(breakpoint)

    pre_covid = returns_df.loc[returns_df.index < breakpoint].copy()
    post_covid = returns_df.loc[returns_df.index >= breakpoint].copy()

    return pre_covid, post_covid

In [ ]:
def compute_skewness(series):
    x = pd.Series(series).dropna()
    mean = x.mean()
    std = x.std(ddof=0)

    if std == 0:
        return np.nan

    return np.mean(((x - mean) / std) ** 3)


def compute_excess_kurtosis(series):
    x = pd.Series(series).dropna()
    mean = x.mean()
    std = x.std(ddof=0)

    if std == 0:
        return np.nan

    kurtosis = np.mean(((x - mean) / std) ** 4)

    return kurtosis - 3


def compute_jarque_bera(series):
    x = pd.Series(series).dropna()
    t = len(x)

    s = compute_skewness(x)
    k = compute_excess_kurtosis(x)

    jb_stat = t * ((s**2) / 6 + (k**2) / 24)
    jb_pvalue = np.exp(-jb_stat / 2)

    return jb_stat, jb_pvalue


def compute_risk_metrics(returns_df, annualization=252):
    rows = []

    for col in returns_df.columns:
        series = returns_df[col].dropna()

        var_5 = series.quantile(0.05)
        cvar_5 = series[series <= var_5].mean()

        jb_stat, jb_pvalue = compute_jarque_bera(series)

        rows.append(
            {
                "Asset": col,
                "Mean_ann": series.mean() * annualization,
                "Vol_ann": series.std() * np.sqrt(annualization),
                "Skewness": compute_skewness(series),
                "Excess_Kurtosis": compute_excess_kurtosis(series),
                "VaR_5": var_5,
                "CVaR_5": cvar_5,
                "JB_stat": jb_stat,
                "JB_pvalue": jb_pvalue,
            }
        )

    result = pd.DataFrame(rows)
    result = result.set_index("Asset")

    return result


def compare_pre_post_risk(pre_covid, post_covid):
    pre_metrics = compute_risk_metrics(pre_covid)
    post_metrics = compute_risk_metrics(post_covid)

    comparison = pd.DataFrame(index=pre_metrics.index)

    comparison["Vol_ann_pre"] = pre_metrics["Vol_ann"]
    comparison["Vol_ann_post"] = post_metrics["Vol_ann"]
    comparison["Vol_change"] = comparison["Vol_ann_post"] - comparison["Vol_ann_pre"]
    comparison["Vol_change_%"] = comparison["Vol_change"] / comparison["Vol_ann_pre"]

    comparison["Skew_pre"] = pre_metrics["Skewness"]
    comparison["Skew_post"] = post_metrics["Skewness"]

    comparison["Excess_Kurtosis_pre"] = pre_metrics["Excess_Kurtosis"]
    comparison["Excess_Kurtosis_post"] = post_metrics["Excess_Kurtosis"]

    comparison["VaR_5_pre"] = pre_metrics["VaR_5"]
    comparison["VaR_5_post"] = post_metrics["VaR_5"]

    comparison["CVaR_5_pre"] = pre_metrics["CVaR_5"]
    comparison["CVaR_5_post"] = post_metrics["CVaR_5"]

    return comparison

In [ ]:
def compute_correlation_matrix(returns_df):
    return returns_df.corr()


def compare_pre_post_correlations(pre_covid, post_covid):
    corr_pre = compute_correlation_matrix(pre_covid)
    corr_post = compute_correlation_matrix(post_covid)
    corr_diff = corr_post - corr_pre

    return corr_pre, corr_post, corr_diff


def average_pairwise_correlation(corr_matrix):
    n = corr_matrix.shape[0]

    total_sum = corr_matrix.values.sum()
    diagonal_sum = n

    average_corr = (total_sum - diagonal_sum) / (n * (n - 1))

    return average_corr


def summarize_correlation_change(corr_pre, corr_post):
    avg_pre = average_pairwise_correlation(corr_pre)
    avg_post = average_pairwise_correlation(corr_post)

    summary = pd.DataFrame(
        {
            "Average_correlation_pre": [avg_pre],
            "Average_correlation_post": [avg_post],
            "Change": [avg_post - avg_pre],
        }
    )

    return summary


def average_block_correlation(corr_matrix, assets_1, assets_2=None):
    if assets_2 is None:
        sub_corr = corr_matrix.loc[assets_1, assets_1]

        n = len(assets_1)
        if n <= 1:
            return float("nan")

        total_sum = sub_corr.values.sum()
        diagonal_sum = n

        return (total_sum - diagonal_sum) / (n * (n - 1))

    sub_corr = corr_matrix.loc[assets_1, assets_2]
    return sub_corr.values.mean()


def summarize_block_correlations(corr_pre, corr_post):
    equities = ["S&P500", "Eurostoxx 50", "Hang Seng", "MSCI EM", "SMI"]
    yields = ["US T 10-year Yield", "German Gov 10-year yield"]
    commodities = ["Oil futures", "Gold"]
    fx = ["EURUSD", "USDJPY", "USDCHF"]
    credit = ["US IG Bonds", "US HY Bonds"]

    blocks = {
        "Equities within": (equities, None),
        "Yields within": (yields, None),
        "Commodities within": (commodities, None),
        "FX within": (fx, None),
        "Credit within": (credit, None),
        "Equities vs Yields": (equities, yields),
        "Equities vs Credit": (equities, credit),
        "Equities vs Commodities": (equities, commodities),
        "Equities vs FX": (equities, fx),
        "Credit vs Yields": (credit, yields),
    }

    rows = []

    for name, (assets_1, assets_2) in blocks.items():
        pre_value = average_block_correlation(corr_pre, assets_1, assets_2)
        post_value = average_block_correlation(corr_post, assets_1, assets_2)

        rows.append(
            {
                "Block": name,
                "Correlation_pre": pre_value,
                "Correlation_post": post_value,
                "Change": post_value - pre_value,
            }
        )

    return pd.DataFrame(rows).set_index("Block")

In [ ]:
def standardize_data(returns_df):
    return (returns_df - returns_df.mean()) / returns_df.std()


def compute_pca(returns_df):
    standardized = standardize_data(returns_df)

    corr_matrix = standardized.corr()

    eigenvalues, eigenvectors = np.linalg.eigh(corr_matrix)

    sorted_indices = np.argsort(eigenvalues)[::-1]

    eigenvalues = eigenvalues[sorted_indices]
    eigenvectors = eigenvectors[:, sorted_indices]

    explained_ratio = eigenvalues / eigenvalues.sum()
    cumulative_ratio = np.cumsum(explained_ratio)

    component_names = [f"PC{i+1}" for i in range(len(eigenvalues))]

    explained_variance = pd.DataFrame(
        {
            "Eigenvalue": eigenvalues,
            "Explained_variance_ratio": explained_ratio,
            "Cumulative_explained_variance": cumulative_ratio,
        },
        index=component_names,
    )

    loadings = pd.DataFrame(
        eigenvectors,
        index=returns_df.columns,
        columns=component_names,
    )

    return explained_variance, loadings


def compare_pre_post_pca(pre_covid, post_covid):
    explained_pre, loadings_pre = compute_pca(pre_covid)
    explained_post, loadings_post = compute_pca(post_covid)

    comparison = pd.DataFrame(
        {
            "Explained_pre": explained_pre["Explained_variance_ratio"],
            "Explained_post": explained_post["Explained_variance_ratio"],
            "Change": (
                explained_post["Explained_variance_ratio"]
                - explained_pre["Explained_variance_ratio"]
            ),
        }
    )

    return explained_pre, explained_post, comparison, loadings_pre, loadings_post

In [ ]:
def compute_rolling_volatility(returns_df, window=252, annualization=252):
    rolling_vol = returns_df.rolling(window=window).std() * (annualization ** 0.5)

    return rolling_vol


def compute_rolling_average_correlation(returns_df, window=252):
    values = []
    dates = []

    for i in range(window, len(returns_df) + 1):
        window_data = returns_df.iloc[i - window:i]
        corr = window_data.corr()

        n = corr.shape[0]
        avg_corr = (corr.values.sum() - n) / (n * (n - 1))

        values.append(avg_corr)
        dates.append(returns_df.index[i - 1])

    return pd.Series(values, index=dates, name="Rolling_average_correlation")

In [ ]:
def run_breakpoint_robustness_clean(returns_df, breakpoints):
    """
    Robustness check across alternative COVID breakpoints.

    The volatility section is separated by economic unit:
    - return-like assets are measured as log returns;
    - yield assets are measured as basis-point changes;
    - oil is treated separately because of the April 2020 negative-price episode.
    This avoids averaging volatility changes across incompatible units.
    """
    rows = []

    for breakpoint in breakpoints:
        pre, post = split_pre_post_covid(returns_df, breakpoint=breakpoint)

        risk_comparison = compare_pre_post_risk(pre, post)

        corr_pre, corr_post, _ = compare_pre_post_correlations(pre, post)
        corr_summary = summarize_correlation_change(corr_pre, corr_post)

        _, _, pca_comparison, _, _ = compare_pre_post_pca(pre, post)

        return_assets = [asset for asset in RETURN_LIKE_ASSETS if asset in risk_comparison.index]
        yield_assets = [asset for asset in YIELD_CHANGE_ASSETS if asset in risk_comparison.index]

        rows.append(
            {
                "Breakpoint": breakpoint,
                "Pre_obs": len(pre),
                "Post_obs": len(post),
                "Avg_return_asset_vol_change": risk_comparison.loc[return_assets, "Vol_change"].mean(),
                "Avg_yield_change_vol_change": risk_comparison.loc[yield_assets, "Vol_change"].mean(),
                "Oil_vol_change": risk_comparison.loc["Oil futures", "Vol_change"] if "Oil futures" in risk_comparison.index else np.nan,
                "Average_corr_pre": corr_summary.loc[0, "Average_correlation_pre"],
                "Average_corr_post": corr_summary.loc[0, "Average_correlation_post"],
                "Average_corr_change": corr_summary.loc[0, "Change"],
                "PC1_pre": pca_comparison.loc["PC1", "Explained_pre"],
                "PC1_post": pca_comparison.loc["PC1", "Explained_post"],
                "PC1_change": pca_comparison.loc["PC1", "Change"],
                "PC1_PC2_pre": pca_comparison.loc[["PC1", "PC2"], "Explained_pre"].sum(),
                "PC1_PC2_post": pca_comparison.loc[["PC1", "PC2"], "Explained_post"].sum(),
                "PC1_PC2_change": pca_comparison.loc[["PC1", "PC2"], "Change"].sum(),
            }
        )

    return pd.DataFrame(rows).set_index("Breakpoint")


def run_no_oil_robustness(returns_df, breakpoint="2020-03-11"):
    returns_no_oil = returns_df.drop(columns=["Oil futures"])

    pre, post = split_pre_post_covid(returns_no_oil, breakpoint=breakpoint)

    risk_comparison = compare_pre_post_risk(pre, post)

    corr_pre, corr_post, corr_diff = compare_pre_post_correlations(pre, post)
    corr_summary = summarize_correlation_change(corr_pre, corr_post)

    explained_pre, explained_post, pca_comparison, loadings_pre, loadings_post = (
        compare_pre_post_pca(pre, post)
    )

    return {
        "risk_comparison_no_oil": risk_comparison,
        "correlation_summary_no_oil": corr_summary,
        "correlation_difference_no_oil": corr_diff,
        "pca_comparison_no_oil": pca_comparison,
        "pca_loadings_pre_no_oil": loadings_pre,
        "pca_loadings_post_no_oil": loadings_post,
    }

In [ ]:
def plot_volatility_change(risk_comparison):
    data = risk_comparison["Vol_change"].sort_values()

    plt.figure(figsize=(12, 7))
    data.plot(kind="barh")

    plt.axvline(0, linewidth=1)
    plt.title("Change in Annualized Risk After COVID-19")
    plt.xlabel("Post-COVID annualized volatility minus pre-COVID annualized volatility")
    plt.ylabel("Asset")

    plt.tight_layout()
    plt.show()


def plot_block_correlation_change(block_corr_summary):
    data = block_corr_summary["Change"].sort_values()

    plt.figure(figsize=(12, 7))
    data.plot(kind="barh")

    plt.axvline(0, linewidth=1)
    plt.title("Change in Asset-Class Correlations After COVID-19")
    plt.xlabel("Post-COVID correlation minus pre-COVID correlation")
    plt.ylabel("Correlation block")

    plt.tight_layout()
    plt.show()


def plot_correlation_difference_heatmap(corr_diff):
    plt.figure(figsize=(12, 10))

    plt.imshow(corr_diff, aspect="auto")
    plt.colorbar(label="Correlation difference")

    plt.xticks(range(len(corr_diff.columns)), corr_diff.columns, rotation=90)
    plt.yticks(range(len(corr_diff.index)), corr_diff.index)

    plt.title("Correlation Matrix Difference: Post-COVID minus Pre-COVID")

    plt.tight_layout()
    plt.show()


def plot_pca_explained_variance(pca_comparison, n_components=5):
    data = pca_comparison.iloc[:n_components][["Explained_pre", "Explained_post"]]

    data.plot(kind="bar", figsize=(10, 6))

    plt.title("PCA: Explained Variance Before and After COVID-19")
    plt.xlabel("Principal component")
    plt.ylabel("Explained variance ratio")
    plt.xticks(rotation=0)

    plt.tight_layout()
    plt.show()


def plot_rolling_volatility(rolling_vol, assets):
    plt.figure(figsize=(12, 7))

    for asset in assets:
        plt.plot(rolling_vol.index, rolling_vol[asset], label=asset)

    plt.axvline(pd.to_datetime("2020-03-11"), linewidth=1, linestyle="--")
    plt.title("Rolling Annualized Volatility")
    plt.xlabel("Date")
    plt.ylabel("Annualized volatility")
    plt.legend()

    plt.tight_layout()
    plt.show()


def plot_rolling_average_correlation(rolling_corr):
    plt.figure(figsize=(12, 6))

    plt.plot(rolling_corr.index, rolling_corr.values)

    plt.axvline(pd.to_datetime("2020-03-11"), linewidth=1, linestyle="--")
    plt.title("Rolling Average Cross-Asset Correlation")
    plt.xlabel("Date")
    plt.ylabel("Average pairwise correlation")

    plt.tight_layout()
    plt.show()


In [ ]:
print("All helper functions loaded successfully.")

In [ ]:
# -----------------------------------------------------------------------------
# Formal inference and portfolio helper functions
# -----------------------------------------------------------------------------

RANDOM_SEED = 42
# 300 bootstrap repetitions keeps the notebook reproducible and fast enough for grading.
# Increase to 1000 for a slower, higher-precision robustness run if desired.
N_BOOTSTRAP = 300

# Return-like investable series used for portfolio exercises and grouped robustness.
# Government yields are excluded from portfolio optimization because they are basis-point changes, not returns.
# Oil is excluded from the baseline portfolio exercise because its April 2020 negative-price episode requires a special transformation.
RETURN_LIKE_ASSETS = [
    "S&P500",
    "Eurostoxx 50",
    "Hang Seng",
    "MSCI EM",
    "SMI",
    "Gold",
    "EURUSD",
    "USDJPY",
    "USDCHF",
    "US IG Bonds",
    "US HY Bonds",
]

YIELD_CHANGE_ASSETS = [
    "US T 10-year Yield",
    "German Gov 10-year yield",
]

INVESTABLE_RETURN_ASSETS = RETURN_LIKE_ASSETS.copy()


def safe_norm_cdf(x):
    """Standard normal CDF using only the Python math library."""
    return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))


def bootstrap_resample_1d(x, rng):
    """Simple bootstrap sample for one series."""
    x = pd.Series(x).dropna()
    if len(x) == 0:
        return x
    idx = rng.integers(0, len(x), size=len(x))
    return x.iloc[idx].reset_index(drop=True)


def bootstrap_resample_rows(df, rng):
    """Resample rows of a multivariate return matrix to preserve cross-sectional dependence."""
    clean = df.dropna()
    idx = rng.integers(0, len(clean), size=len(clean))
    return clean.iloc[idx].reset_index(drop=True)


def historical_var(x, alpha=0.05):
    """Historical Value-at-Risk as a positive loss number."""
    x = pd.Series(x).dropna()
    if len(x) == 0:
        return np.nan
    return -np.quantile(x, alpha)


def historical_cvar(x, alpha=0.05):
    """Historical Conditional Value-at-Risk as a positive average tail-loss number."""
    x = pd.Series(x).dropna()
    if len(x) == 0:
        return np.nan
    threshold = np.quantile(x, alpha)
    return -x[x <= threshold].mean()


def annualized_std(x, annualization=252):
    x = pd.Series(x).dropna()
    return x.std(ddof=1) * np.sqrt(annualization)


def metric_change_with_bootstrap(pre, post, metric_func, n_bootstrap=N_BOOTSTRAP, seed=RANDOM_SEED):
    """
    Estimate post-minus-pre metric change with bootstrap confidence interval.
    The p-value is the two-sided bootstrap sign probability around zero.
    """
    pre = pd.Series(pre).dropna()
    post = pd.Series(post).dropna()

    observed_pre = metric_func(pre)
    observed_post = metric_func(post)
    observed_change = observed_post - observed_pre

    rng = np.random.default_rng(seed)
    boot_changes = []

    for _ in range(n_bootstrap):
        pre_b = bootstrap_resample_1d(pre, rng)
        post_b = bootstrap_resample_1d(post, rng)
        boot_changes.append(metric_func(pd.Series(post_b)) - metric_func(pd.Series(pre_b)))

    boot_changes = np.asarray(boot_changes, dtype=float)
    ci_low, ci_high = np.nanpercentile(boot_changes, [2.5, 97.5])

    p_left = np.nanmean(boot_changes <= 0)
    p_right = np.nanmean(boot_changes >= 0)
    p_value = min(1.0, 2.0 * min(p_left, p_right))

    return observed_pre, observed_post, observed_change, ci_low, ci_high, p_value


def classify_change(change, p_value, positive_word="increased", negative_word="decreased"):
    if pd.isna(p_value):
        return "not available"
    if p_value < 0.05 and change > 0:
        return f"significantly {positive_word}"
    if p_value < 0.05 and change < 0:
        return f"significantly {negative_word}"
    return "not statistically significant"


def bootstrap_risk_level_tests(pre_covid, post_covid, n_bootstrap=N_BOOTSTRAP):
    def bootstrap_skewness(x):
        x = pd.Series(x).dropna()
        mean = x.mean()
        std = x.std(ddof=0)
        if std == 0:
            return np.nan
        return np.mean(((x - mean) / std) ** 3)

    def bootstrap_excess_kurtosis(x):
        x = pd.Series(x).dropna()
        mean = x.mean()
        std = x.std(ddof=0)
        if std == 0:
            return np.nan
        return np.mean(((x - mean) / std) ** 4) - 3

    rows = []
    metrics = {
        "Annualized volatility": annualized_std,
        "VaR 5%": historical_var,
        "CVaR 5%": historical_cvar,
        "Skewness": bootstrap_skewness,
        "Excess kurtosis": bootstrap_excess_kurtosis,
    }

    for asset in pre_covid.columns:
        for metric_name, metric_func in metrics.items():
            pre_v, post_v, change, ci_low, ci_high, p_value = metric_change_with_bootstrap(
                pre_covid[asset],
                post_covid[asset],
                metric_func,
                n_bootstrap=n_bootstrap,
                seed=RANDOM_SEED,
            )
            rows.append(
                {
                    "Asset": asset,
                    "Metric": metric_name,
                    "Pre": pre_v,
                    "Post": post_v,
                    "Change": change,
                    "CI_low": ci_low,
                    "CI_high": ci_high,
                    "p_value": p_value,
                    "Interpretation": classify_change(change, p_value),
                }
            )

    return pd.DataFrame(rows)


def fisher_correlation_test(pre_df, post_df, pairs):
    rows = []

    for asset_1, asset_2 in pairs:
        if asset_1 not in pre_df.columns or asset_2 not in pre_df.columns:
            continue

        pre_pair = pre_df[[asset_1, asset_2]].dropna()
        post_pair = post_df[[asset_1, asset_2]].dropna()
        n_pre, n_post = len(pre_pair), len(post_pair)

        rho_pre = pre_pair[asset_1].corr(pre_pair[asset_2])
        rho_post = post_pair[asset_1].corr(post_pair[asset_2])

        rho_pre_clip = np.clip(rho_pre, -0.999999, 0.999999)
        rho_post_clip = np.clip(rho_post, -0.999999, 0.999999)

        z_pre = 0.5 * np.log((1 + rho_pre_clip) / (1 - rho_pre_clip))
        z_post = 0.5 * np.log((1 + rho_post_clip) / (1 - rho_post_clip))
        se = np.sqrt(1 / (n_pre - 3) + 1 / (n_post - 3))
        z_stat = (z_post - z_pre) / se
        p_value = 2 * (1 - safe_norm_cdf(abs(z_stat)))

        change = rho_post - rho_pre
        interp = classify_change(change, p_value, "increased", "decreased")

        rows.append(
            {
                "Pair": f"{asset_1} vs {asset_2}",
                "Asset_1": asset_1,
                "Asset_2": asset_2,
                "Corr_pre": rho_pre,
                "Corr_post": rho_post,
                "Change": change,
                "z_stat": z_stat,
                "p_value": p_value,
                "Economic interpretation": interp,
            }
        )

    return pd.DataFrame(rows)


def average_block_correlation_from_data(df, assets_a, assets_b):
    available_a = [a for a in assets_a if a in df.columns]
    available_b = [b for b in assets_b if b in df.columns]

    if len(available_a) == 0 or len(available_b) == 0:
        return np.nan

    corr = df[available_a + available_b].corr()
    values = []

    for a in available_a:
        for b in available_b:
            if a == b:
                continue
            # avoid double-counting within same block
            if set(available_a) == set(available_b) and available_a.index(a) >= available_b.index(b):
                continue
            values.append(corr.loc[a, b])

    return np.nanmean(values) if len(values) else np.nan


def bootstrap_block_correlation_tests(pre_df, post_df, block_pairs, n_bootstrap=N_BOOTSTRAP):
    rows = []
    rng = np.random.default_rng(RANDOM_SEED)

    for block_name, assets_a, assets_b in block_pairs:
        pre_value = average_block_correlation_from_data(pre_df, assets_a, assets_b)
        post_value = average_block_correlation_from_data(post_df, assets_a, assets_b)
        observed_change = post_value - pre_value

        boot_changes = []
        for _ in range(n_bootstrap):
            pre_b = bootstrap_resample_rows(pre_df, rng)
            post_b = bootstrap_resample_rows(post_df, rng)
            boot_changes.append(
                average_block_correlation_from_data(post_b, assets_a, assets_b)
                - average_block_correlation_from_data(pre_b, assets_a, assets_b)
            )

        boot_changes = np.asarray(boot_changes, dtype=float)
        ci_low, ci_high = np.nanpercentile(boot_changes, [2.5, 97.5])
        p_value = min(1.0, 2.0 * min(np.nanmean(boot_changes <= 0), np.nanmean(boot_changes >= 0)))

        rows.append(
            {
                "Block pair": block_name,
                "Corr_pre": pre_value,
                "Corr_post": post_value,
                "Change": observed_change,
                "CI_low": ci_low,
                "CI_high": ci_high,
                "p_value": p_value,
                "Interpretation": classify_change(observed_change, p_value, "increased", "decreased"),
            }
        )

    return pd.DataFrame(rows)


def covariance_frobenius_test(pre_df, post_df, n_bootstrap=N_BOOTSTRAP):
    """Bootstrap the change in the full covariance matrix using the Frobenius norm."""
    pre_clean = pre_df.dropna()
    post_clean = post_df.dropna()

    sigma_pre = pre_clean.cov().values
    sigma_post = post_clean.cov().values
    observed = np.linalg.norm(sigma_post - sigma_pre, ord="fro")

    rng = np.random.default_rng(RANDOM_SEED)
    boot_values = []
    for _ in range(n_bootstrap):
        pre_b = bootstrap_resample_rows(pre_clean, rng)
        post_b = bootstrap_resample_rows(post_clean, rng)
        boot_values.append(np.linalg.norm(post_b.cov().values - pre_b.cov().values, ord="fro"))

    boot_values = np.asarray(boot_values)
    ci_low, ci_high = np.nanpercentile(boot_values, [2.5, 97.5])

    return pd.DataFrame(
        [
            {
                "Metric": "Frobenius norm of covariance difference",
                "Value": observed,
                "CI_low": ci_low,
                "CI_high": ci_high,
                "Interpretation": "larger values indicate a stronger global covariance-matrix shift",
            }
        ]
    )



def correlation_frobenius_test(pre_df, post_df, n_bootstrap=N_BOOTSTRAP):
    """
    Global test of correlation-structure change.

    The correlation matrix is unit-free, so this test avoids the unit-mixing problem
    created by combining log returns, yield changes in basis points and transformed oil.
    """
    pre_clean = pre_df.dropna()
    post_clean = post_df.dropna()

    corr_pre = pre_clean.corr().values
    corr_post = post_clean.corr().values
    observed = np.linalg.norm(corr_post - corr_pre, ord="fro")

    rng = np.random.default_rng(RANDOM_SEED)
    boot_values = []
    for _ in range(n_bootstrap):
        pre_b = bootstrap_resample_rows(pre_clean, rng)
        post_b = bootstrap_resample_rows(post_clean, rng)
        corr_pre_b = pre_b.corr().values
        corr_post_b = post_b.corr().values
        boot_values.append(np.linalg.norm(corr_post_b - corr_pre_b, ord="fro"))

    boot_values = np.asarray(boot_values)
    ci_low, ci_high = np.nanpercentile(boot_values, [2.5, 97.5])

    return pd.DataFrame(
        [
            {
                "Metric": "Frobenius norm of correlation-matrix difference",
                "Value": observed,
                "CI_low": ci_low,
                "CI_high": ci_high,
                "Interpretation": "unit-free global measure of cross-asset correlation-structure change",
            }
        ]
    )

def pca_concentration_metrics(df):
    explained, _ = compute_pca(df)
    pc1 = explained.loc["PC1", "Explained_variance_ratio"]
    pc12 = explained.loc[["PC1", "PC2"], "Explained_variance_ratio"].sum()
    return pc1, pc12


def bootstrap_pca_concentration(pre_df, post_df, n_bootstrap=N_BOOTSTRAP):
    pre_pc1, pre_pc12 = pca_concentration_metrics(pre_df)
    post_pc1, post_pc12 = pca_concentration_metrics(post_df)

    rng = np.random.default_rng(RANDOM_SEED)
    pc1_changes = []
    pc12_changes = []

    for _ in range(n_bootstrap):
        pre_b = bootstrap_resample_rows(pre_df, rng)
        post_b = bootstrap_resample_rows(post_df, rng)
        pre_b_pc1, pre_b_pc12 = pca_concentration_metrics(pre_b)
        post_b_pc1, post_b_pc12 = pca_concentration_metrics(post_b)
        pc1_changes.append(post_b_pc1 - pre_b_pc1)
        pc12_changes.append(post_b_pc12 - pre_b_pc12)

    rows = []
    for metric, pre_value, post_value, boot_changes in [
        ("PC1 explained variance", pre_pc1, post_pc1, np.asarray(pc1_changes)),
        ("PC1 + PC2 explained variance", pre_pc12, post_pc12, np.asarray(pc12_changes)),
    ]:
        change = post_value - pre_value
        ci_low, ci_high = np.nanpercentile(boot_changes, [2.5, 97.5])
        p_value = min(1.0, 2.0 * min(np.nanmean(boot_changes <= 0), np.nanmean(boot_changes >= 0)))
        rows.append(
            {
                "PCA metric": metric,
                "Pre": pre_value,
                "Post": post_value,
                "Change": change,
                "CI_low": ci_low,
                "CI_high": ci_high,
                "p_value": p_value,
                "Interpretation": classify_change(change, p_value, "increased", "decreased"),
            }
        )

    return pd.DataFrame(rows)


def compute_rolling_pair_correlation(df, asset_1, asset_2, window=252):
    return df[asset_1].rolling(window).corr(df[asset_2]).dropna().rename(f"{asset_1} vs {asset_2}")


def segmented_mean_breaks(series, max_breaks=3, min_segment=126, grid_step=21):
    """
    Bai-Perron-style segmented mean-break approximation by dynamic programming.
    Break candidates are evaluated on a business-day grid so the notebook remains
    fast enough to run end-to-end while preserving the regime-shift interpretation.
    """
    y = pd.Series(series).dropna()
    values = y.values
    n = len(values)

    if n < 2 * min_segment:
        return pd.DataFrame(
            [
                {
                    "Segment": 1,
                    "Start": y.index.min(),
                    "End": y.index.max(),
                    "Avg correlation": y.mean(),
                    "Observations": len(y),
                    "Interpretation": "higher values mean weaker equity-bond diversification" if y.mean() > 0 else "negative values mean stronger equity-bond diversification",
                }
            ]
        )

    candidates = list(range(0, n, grid_step))
    if candidates[-1] != n:
        candidates.append(n)
    candidates = sorted(set([0, n] + candidates))
    m = len(candidates)

    csum = np.r_[0, np.cumsum(values)]
    csum2 = np.r_[0, np.cumsum(values ** 2)]

    def segment_sse(start, end):
        length = end - start
        if length < min_segment:
            return np.inf
        segment_sum = csum[end] - csum[start]
        segment_sum2 = csum2[end] - csum2[start]
        return segment_sum2 - segment_sum ** 2 / length

    max_segments = max_breaks + 1
    dp = np.full((max_segments + 1, m), np.inf)
    prev = np.full((max_segments + 1, m), -1, dtype=int)
    dp[0, 0] = 0

    for k in range(1, max_segments + 1):
        for j_idx in range(1, m):
            end = candidates[j_idx]
            best_value = np.inf
            best_i = -1
            for i_idx in range(j_idx):
                start = candidates[i_idx]
                if end - start < min_segment:
                    continue
                value = dp[k - 1, i_idx] + segment_sse(start, end)
                if value < best_value:
                    best_value = value
                    best_i = i_idx
            dp[k, j_idx] = best_value
            prev[k, j_idx] = best_i

    final_idx = m - 1
    best_k = 1
    best_bic = np.inf
    for k in range(1, max_segments + 1):
        residual_sse = dp[k, final_idx]
        if not np.isfinite(residual_sse) or residual_sse <= 0:
            continue
        bic = n * np.log(residual_sse / n) + k * np.log(n)
        if bic < best_bic:
            best_bic = bic
            best_k = k

    segments_idx = []
    j_idx = final_idx
    for k in range(best_k, 0, -1):
        i_idx = prev[k, j_idx]
        if i_idx < 0:
            break
        segments_idx.append((candidates[i_idx], candidates[j_idx]))
        j_idx = i_idx
    segments_idx = list(reversed(segments_idx)) or [(0, n)]

    rows = []
    for i, j in segments_idx:
        segment = y.iloc[i:j]
        rows.append(
            {
                "Segment": len(rows) + 1,
                "Start": segment.index.min(),
                "End": segment.index.max(),
                "Avg correlation": segment.mean(),
                "Observations": len(segment),
                "Interpretation": "higher values mean weaker equity-bond diversification" if segment.mean() > 0 else "negative values mean stronger equity-bond diversification",
            }
        )

    return pd.DataFrame(rows)


def portfolio_volatility(weights, cov_matrix, annualization=252):
    weights = np.asarray(weights)
    return float(np.sqrt(weights @ cov_matrix.values @ weights) * np.sqrt(annualization))


def diversification_ratio(weights, cov_matrix):
    weights = np.asarray(weights)
    asset_vols = np.sqrt(np.diag(cov_matrix.values))
    numerator = weights @ asset_vols
    denominator = np.sqrt(weights @ cov_matrix.values @ weights)
    return float(numerator / denominator)


def min_variance_weights(cov_matrix):
    n = cov_matrix.shape[0]
    x0 = np.repeat(1 / n, n)

    if minimize is None:
        # Fallback: unconstrained fully invested solution, clipped and renormalized.
        inv_cov = np.linalg.pinv(cov_matrix.values)
        ones = np.ones(n)
        w = inv_cov @ ones / (ones @ inv_cov @ ones)
        w = np.clip(w, 0, None)
        return w / w.sum()

    def objective(w):
        return w @ cov_matrix.values @ w

    constraints = ({"type": "eq", "fun": lambda w: np.sum(w) - 1},)
    bounds = [(0, 1) for _ in range(n)]
    result = minimize(objective, x0, method="SLSQP", bounds=bounds, constraints=constraints)

    if not result.success:
        raise RuntimeError(f"Minimum-variance optimization failed: {result.message}")

    return result.x


def risk_contributions(weights, cov_matrix):
    weights = np.asarray(weights)
    sigma_p = np.sqrt(weights @ cov_matrix.values @ weights)
    marginal = cov_matrix.values @ weights / sigma_p
    contributions = weights * marginal
    pct_contributions = contributions / contributions.sum()
    return pct_contributions


def erc_weights(cov_matrix):
    n = cov_matrix.shape[0]
    x0 = np.repeat(1 / n, n)

    if minimize is None:
        # Fallback if scipy is unavailable.
        return x0

    def objective(w):
        rc = risk_contributions(w, cov_matrix)
        target = np.repeat(1 / n, n)
        return np.sum((rc - target) ** 2)

    constraints = ({"type": "eq", "fun": lambda w: np.sum(w) - 1},)
    bounds = [(0.0001, 1) for _ in range(n)]
    result = minimize(objective, x0, method="SLSQP", bounds=bounds, constraints=constraints)

    if not result.success:
        raise RuntimeError(f"ERC optimization failed: {result.message}")

    return result.x


def portfolio_implication_tables(pre_df, post_df):
    cov_pre = pre_df.cov()
    cov_post = post_df.cov()
    assets = pre_df.columns.tolist()
    n = len(assets)

    weights = {
        "Equal weight": np.repeat(1 / n, n),
        "Minimum variance": min_variance_weights(cov_pre),
        "Equal risk contribution": erc_weights(cov_pre),
    }

    summary_rows = []
    weight_rows = []
    rc_rows = []

    for name, w_pre_design in weights.items():
        # Ex-ante weights calibrated on the pre-COVID covariance matrix.
        pre_vol = portfolio_volatility(w_pre_design, cov_pre)
        post_vol_same_weights = portfolio_volatility(w_pre_design, cov_post)
        pre_dr = diversification_ratio(w_pre_design, cov_pre)
        post_dr_same_weights = diversification_ratio(w_pre_design, cov_post)

        summary_rows.append(
            {
                "Portfolio": name,
                "Pre vol": pre_vol,
                "Post vol using same weights": post_vol_same_weights,
                "Vol change": post_vol_same_weights - pre_vol,
                "Pre diversification ratio": pre_dr,
                "Post diversification ratio using same weights": post_dr_same_weights,
                "Interpretation": "tests whether the same allocation became riskier or less diversified after COVID",
            }
        )

        rc_pre = risk_contributions(w_pre_design, cov_pre)
        rc_post = risk_contributions(w_pre_design, cov_post)
        for i, asset in enumerate(assets):
            rc_rows.append(
                {
                    "Portfolio": name,
                    "Asset": asset,
                    "Risk contribution pre": rc_pre[i],
                    "Risk contribution post using same weights": rc_post[i],
                    "Change": rc_post[i] - rc_pre[i],
                }
            )

    # Compare optimal weights recalibrated separately pre and post.
    minvar_pre = min_variance_weights(cov_pre)
    minvar_post = min_variance_weights(cov_post)
    erc_pre = erc_weights(cov_pre)
    erc_post = erc_weights(cov_post)

    for i, asset in enumerate(assets):
        weight_rows.append(
            {
                "Asset": asset,
                "Min-var weight pre": minvar_pre[i],
                "Min-var weight post": minvar_post[i],
                "ERC weight pre": erc_pre[i],
                "ERC weight post": erc_post[i],
            }
        )

    return pd.DataFrame(summary_rows), pd.DataFrame(weight_rows), pd.DataFrame(rc_rows)


def run_no_yields_robustness(returns_df, breakpoint="2020-03-11"):
    no_yields_df = returns_df.drop(columns=[c for c in YIELD_COLUMNS if c in returns_df.columns])
    pre, post = split_pre_post_covid(no_yields_df, breakpoint=breakpoint)
    corr_pre, corr_post, _ = compare_pre_post_correlations(pre, post)
    _, _, pca_comparison, _, _ = compare_pre_post_pca(pre, post)
    return {
        "risk_comparison_no_yields": compare_pre_post_risk(pre, post),
        "correlation_summary_no_yields": summarize_correlation_change(corr_pre, corr_post),
        "pca_comparison_no_yields": pca_comparison,
    }


def run_weekly_robustness(returns_df, breakpoint="2020-03-11"):
    weekly_df = returns_df.resample("W-FRI").sum().dropna()
    pre, post = split_pre_post_covid(weekly_df, breakpoint=breakpoint)
    corr_pre, corr_post, _ = compare_pre_post_correlations(pre, post)
    _, _, pca_comparison, _, _ = compare_pre_post_pca(pre, post)
    return {
        "risk_comparison_weekly": compare_pre_post_risk(pre, post),
        "correlation_summary_weekly": summarize_correlation_change(corr_pre, corr_post),
        "pca_comparison_weekly": pca_comparison,
    }


# Faster NumPy implementation overriding the pandas-heavy version above.
def _average_block_correlation_numpy(values, columns, assets_a, assets_b):
    idx_a = [columns.index(a) for a in assets_a if a in columns]
    idx_b = [columns.index(b) for b in assets_b if b in columns]
    if len(idx_a) == 0 or len(idx_b) == 0:
        return np.nan
    corr = np.corrcoef(values, rowvar=False)
    vals = []
    same_block = set(idx_a) == set(idx_b)
    for ia in idx_a:
        for ib in idx_b:
            if ia == ib:
                continue
            if same_block and ia >= ib:
                continue
            vals.append(corr[ia, ib])
    return float(np.nanmean(vals)) if vals else np.nan


def bootstrap_block_correlation_tests(pre_df, post_df, block_pairs, n_bootstrap=N_BOOTSTRAP):
    """Fast bootstrap tests for average block-correlation changes."""
    rows = []
    rng = np.random.default_rng(RANDOM_SEED)

    pre_clean = pre_df.dropna()
    post_clean = post_df.dropna()
    columns = pre_clean.columns.tolist()
    pre_values = pre_clean.values
    post_values = post_clean.values

    for block_name, assets_a, assets_b in block_pairs:
        pre_value = _average_block_correlation_numpy(pre_values, columns, assets_a, assets_b)
        post_value = _average_block_correlation_numpy(post_values, columns, assets_a, assets_b)
        observed_change = post_value - pre_value

        boot_changes = np.empty(n_bootstrap)
        for b in range(n_bootstrap):
            pre_idx = rng.integers(0, len(pre_values), size=len(pre_values))
            post_idx = rng.integers(0, len(post_values), size=len(post_values))
            pre_b = pre_values[pre_idx, :]
            post_b = post_values[post_idx, :]
            boot_changes[b] = (
                _average_block_correlation_numpy(post_b, columns, assets_a, assets_b)
                - _average_block_correlation_numpy(pre_b, columns, assets_a, assets_b)
            )

        ci_low, ci_high = np.nanpercentile(boot_changes, [2.5, 97.5])
        p_value = min(1.0, 2.0 * min(np.nanmean(boot_changes <= 0), np.nanmean(boot_changes >= 0)))

        rows.append(
            {
                "Block pair": block_name,
                "Corr_pre": pre_value,
                "Corr_post": post_value,
                "Change": observed_change,
                "CI_low": ci_low,
                "CI_high": ci_high,
                "p_value": p_value,
                "Interpretation": classify_change(observed_change, p_value, "increased", "decreased"),
            }
        )

    return pd.DataFrame(rows)


# Faster vectorized implementation overriding the pandas-heavy risk bootstrap above.
def _vectorized_metric(samples, metric_name):
    if metric_name == "Annualized volatility":
        return np.std(samples, axis=1, ddof=1) * np.sqrt(252)
    if metric_name == "VaR 5%":
        return -np.quantile(samples, 0.05, axis=1)
    if metric_name == "CVaR 5%":
        q = np.quantile(samples, 0.05, axis=1)
        mask = samples <= q[:, None]
        return -np.sum(np.where(mask, samples, 0.0), axis=1) / np.sum(mask, axis=1)
    if metric_name == "Skewness":
        mean = np.mean(samples, axis=1)
        std = np.std(samples, axis=1, ddof=0)
        z = (samples - mean[:, None]) / std[:, None]
        return np.mean(z ** 3, axis=1)
    if metric_name == "Excess kurtosis":
        mean = np.mean(samples, axis=1)
        std = np.std(samples, axis=1, ddof=0)
        z = (samples - mean[:, None]) / std[:, None]
        return np.mean(z ** 4, axis=1) - 3
    raise ValueError(metric_name)


def _observed_metric(arr, metric_name):
    arr = np.asarray(arr, dtype=float)
    if metric_name == "Annualized volatility":
        return np.std(arr, ddof=1) * np.sqrt(252)
    if metric_name == "VaR 5%":
        return -np.quantile(arr, 0.05)
    if metric_name == "CVaR 5%":
        q = np.quantile(arr, 0.05)
        return -np.mean(arr[arr <= q])
    if metric_name == "Skewness":
        return compute_skewness(arr)
    if metric_name == "Excess kurtosis":
        return compute_excess_kurtosis(arr)
    raise ValueError(metric_name)


def bootstrap_risk_level_tests(pre_covid, post_covid, n_bootstrap=N_BOOTSTRAP):
    """Fast vectorized bootstrap tests for risk-level changes."""
    rows = []
    metric_names = ["Annualized volatility", "VaR 5%", "CVaR 5%", "Skewness", "Excess kurtosis"]
    rng = np.random.default_rng(RANDOM_SEED)

    for asset in pre_covid.columns:
        pre_arr = pd.Series(pre_covid[asset]).dropna().to_numpy(dtype=float)
        post_arr = pd.Series(post_covid[asset]).dropna().to_numpy(dtype=float)

        pre_idx = rng.integers(0, len(pre_arr), size=(n_bootstrap, len(pre_arr)))
        post_idx = rng.integers(0, len(post_arr), size=(n_bootstrap, len(post_arr)))
        pre_samples = pre_arr[pre_idx]
        post_samples = post_arr[post_idx]

        for metric_name in metric_names:
            observed_pre = _observed_metric(pre_arr, metric_name)
            observed_post = _observed_metric(post_arr, metric_name)
            observed_change = observed_post - observed_pre

            pre_boot = _vectorized_metric(pre_samples, metric_name)
            post_boot = _vectorized_metric(post_samples, metric_name)
            boot_changes = post_boot - pre_boot

            ci_low, ci_high = np.nanpercentile(boot_changes, [2.5, 97.5])
            p_left = np.nanmean(boot_changes <= 0)
            p_right = np.nanmean(boot_changes >= 0)
            p_value = min(1.0, 2.0 * min(p_left, p_right))

            rows.append(
                {
                    "Asset": asset,
                    "Metric": metric_name,
                    "Pre": observed_pre,
                    "Post": observed_post,
                    "Change": observed_change,
                    "CI_low": ci_low,
                    "CI_high": ci_high,
                    "p_value": p_value,
                    "Interpretation": classify_change(observed_change, p_value),
                }
            )

    return pd.DataFrame(rows)


# Faster NumPy implementation overriding the pandas PCA-bootstrap version above.
def _pca_concentration_metrics_numpy(values):
    x = np.asarray(values, dtype=float)
    x = (x - x.mean(axis=0)) / x.std(axis=0, ddof=1)
    corr = np.corrcoef(x, rowvar=False)
    eigenvalues = np.linalg.eigvalsh(corr)[::-1]
    ratios = eigenvalues / eigenvalues.sum()
    return float(ratios[0]), float(ratios[:2].sum())


def bootstrap_pca_concentration(pre_df, post_df, n_bootstrap=N_BOOTSTRAP):
    """Fast bootstrap tests for PC1 and PC1+PC2 concentration changes."""
    pre_clean = pre_df.dropna()
    post_clean = post_df.dropna()
    pre_values = pre_clean.values
    post_values = post_clean.values

    pre_pc1, pre_pc12 = _pca_concentration_metrics_numpy(pre_values)
    post_pc1, post_pc12 = _pca_concentration_metrics_numpy(post_values)

    rng = np.random.default_rng(RANDOM_SEED)
    pc1_changes = np.empty(n_bootstrap)
    pc12_changes = np.empty(n_bootstrap)

    for b in range(n_bootstrap):
        pre_idx = rng.integers(0, len(pre_values), size=len(pre_values))
        post_idx = rng.integers(0, len(post_values), size=len(post_values))
        pre_b_pc1, pre_b_pc12 = _pca_concentration_metrics_numpy(pre_values[pre_idx, :])
        post_b_pc1, post_b_pc12 = _pca_concentration_metrics_numpy(post_values[post_idx, :])
        pc1_changes[b] = post_b_pc1 - pre_b_pc1
        pc12_changes[b] = post_b_pc12 - pre_b_pc12

    rows = []
    for metric, pre_value, post_value, boot_changes in [
        ("PC1 explained variance", pre_pc1, post_pc1, pc1_changes),
        ("PC1 + PC2 explained variance", pre_pc12, post_pc12, pc12_changes),
    ]:
        change = post_value - pre_value
        ci_low, ci_high = np.nanpercentile(boot_changes, [2.5, 97.5])
        p_value = min(1.0, 2.0 * min(np.nanmean(boot_changes <= 0), np.nanmean(boot_changes >= 0)))
        rows.append(
            {
                "PCA metric": metric,
                "Pre": pre_value,
                "Post": post_value,
                "Change": change,
                "CI_low": ci_low,
                "CI_high": ci_high,
                "p_value": p_value,
                "Interpretation": classify_change(change, p_value, "increased", "decreased"),
            }
        )

    return pd.DataFrame(rows)


## 1. Data loading

The raw dataset is loaded from `Data.xlsx`. The first step is to inspect the available variables, the date range and missing values.

In [ ]:
raw_df = load_raw_data(DATA_PATH)

print("Raw data shape:", raw_df.shape)
print("Start date:", raw_df.index.min())
print("End date:", raw_df.index.max())

raw_df.head()

In [ ]:
missing_values = raw_df.isna().sum().sort_values(ascending=False)
missing_values

## 2. Return transformations and sample split

Following the EMiF course, prices and indices are transformed into returns because financial returns are more suitable for empirical analysis than price levels. Most price-like series are transformed into log returns. Government bond yields are transformed into daily changes in basis points.

Oil futures require special treatment because WTI futures became negative in April 2020. Therefore, the oil series is transformed using the inverse hyperbolic sine difference, which is defined for negative values.

In [ ]:
returns_df = prepare_returns(raw_df)

print("Returns data shape:", returns_df.shape)
print("Start date:", returns_df.index.min())
print("End date:", returns_df.index.max())

returns_df.head()

In [ ]:
pre_covid, post_covid = split_pre_post_covid(
    returns_df,
    breakpoint="2020-03-11",
)

print("Pre-COVID shape:", pre_covid.shape)
print("Pre-COVID period:", pre_covid.index.min(), "to", pre_covid.index.max())

print("Post-COVID shape:", post_covid.shape)
print("Post-COVID period:", post_covid.index.min(), "to", post_covid.index.max())

### Data and transformations table

The table below documents the treatment of the raw variables. This is important because the project guidelines require the notebook to explain the transformations, missing-value handling, sample split and limitations. The key interpretation rule is that return-like assets and yield-change assets are not compared in levels; the analysis compares their pre/post changes within each series.


In [ ]:
data_treatment_table = pd.DataFrame(
    [
        {"Item": "Frequency", "Choice": "Daily observations from Data.xlsx"},
        {"Item": "Sample", "Choice": f"{returns_df.index.min().date()} to {returns_df.index.max().date()} after transformations"},
        {"Item": "COVID breakpoint", "Choice": "11 March 2020, WHO pandemic announcement and major global market stress point"},
        {"Item": "Price/index/FX/bond-price series", "Choice": "Log returns"},
        {"Item": "Government yields", "Choice": "Daily changes in basis points"},
        {"Item": "Oil futures", "Choice": "asinh difference to handle the April 2020 negative WTI episode"},
        {"Item": "Missing values", "Choice": "Balanced panel after transformations using dropna()"},
        {"Item": "Reason for balanced panel", "Choice": "Required for comparable correlations, covariance matrices and PCA"},
        {"Item": "Interpretation rule", "Choice": "Do not compare yield-change volatility levels directly with return volatility; compare pre/post changes within each asset"},
    ]
)

print("Data treatment table:")
display(data_treatment_table)


## 3. Risk levels: pre-COVID vs post-COVID

This section compares the level and distribution of risk before and after COVID-19.

We compute the following metrics for each asset:

- Annualized volatility
- Skewness
- Excess kurtosis
- Historical 5% Value-at-Risk
- Historical 5% Conditional Value-at-Risk

These indicators are linked to the EMiF course discussion of financial return moments, non-normality, fat tails and volatility. They allow us to test whether the level and shape of asset risk changed after COVID-19.

In [ ]:
risk_comparison = compare_pre_post_risk(pre_covid, post_covid)

risk_comparison_rounded = risk_comparison.round(4)
risk_comparison_rounded

In [ ]:
print("Risk comparison table:")
display(risk_comparison_rounded)


## 3.1 Statistical tests for changes in risk levels

The descriptive statistics above are useful, but the project needs formal evidence. Because financial returns are non-normal and fat-tailed, we use bootstrap confidence intervals and two-sided bootstrap p-values for pre/post changes in volatility, VaR, CVaR, skewness and excess kurtosis.

Positive VaR and CVaR numbers are interpreted as losses. A positive CVaR change therefore means that the average 5% tail loss became larger after COVID.


In [ ]:
# Keep this inference section robust to both pandas Series and NumPy arrays.
def compute_skewness(series):
    x = pd.Series(series).dropna()
    mean = x.mean()
    std = x.std(ddof=0)
    if std == 0:
        return np.nan
    return np.mean(((x - mean) / std) ** 3)


def compute_excess_kurtosis(series):
    x = pd.Series(series).dropna()
    mean = x.mean()
    std = x.std(ddof=0)
    if std == 0:
        return np.nan
    return np.mean(((x - mean) / std) ** 4) - 3


def bootstrap_resample_1d(x, rng):
    x = pd.Series(x).dropna()
    if len(x) == 0:
        return x
    idx = rng.integers(0, len(x), size=len(x))
    return x.iloc[idx].reset_index(drop=True)


risk_level_tests = bootstrap_risk_level_tests(
    pre_covid,
    post_covid,
    n_bootstrap=N_BOOTSTRAP,
)

volatility_test_table = risk_level_tests.query("Metric == 'Annualized volatility'").copy()
cvar_test_table = risk_level_tests.query("Metric == 'CVaR 5%'").copy()
var_test_table = risk_level_tests.query("Metric == 'VaR 5%'").copy()

print("Bootstrap volatility-change tests:")
display(volatility_test_table.round(4))

print("Bootstrap CVaR-change tests:")
display(cvar_test_table.round(4))


In [ ]:
vol_changes = risk_comparison["Vol_change"].sort_values(ascending=False)

print("Largest increases in annualized risk:")
display(vol_changes.head(7).round(4))

print("Largest decreases in annualized risk:")
display(vol_changes.tail(7).round(4))

### Figure: change in annualized volatility

The following figure shows the difference between post-COVID and pre-COVID annualized volatility. Positive values mean that realized risk increased after COVID-19.

In [ ]:
plot_volatility_change(risk_comparison)


### Interpretation

The volatility results suggest that the post-COVID change in risk levels was not uniform across asset classes.

Risk increased particularly for oil futures, government yield changes, credit bonds, gold and the S&P500. This indicates that COVID-19 did not only affect equity risk, but also energy risk, rates risk and credit risk.

However, several assets show lower or broadly stable annualized volatility after COVID-19, including some equity indices and FX rates. Therefore, the evidence does not support the idea that all markets simply became riskier after COVID-19.

The first conclusion is therefore partial: risk levels changed after COVID-19, but the change was selective across asset classes.

## 4. Risk co-movement: correlations and diversification

This section studies whether cross-asset co-movements changed after COVID-19.

Correlation analysis helps answer whether diversification benefits changed. If correlations rise after COVID, assets move more together and diversification becomes less effective. If correlations fall or change selectively, the structure of risk changes in a more nuanced way.

We compute:

1. Pre-COVID correlation matrix.
2. Post-COVID correlation matrix.
3. Difference matrix: post-COVID minus pre-COVID.
4. Average pairwise correlation.
5. Asset-class block correlations.

In [ ]:
corr_pre, corr_post, corr_diff = compare_pre_post_correlations(
    pre_covid,
    post_covid,
)

corr_summary = summarize_correlation_change(corr_pre, corr_post)
block_corr_summary = summarize_block_correlations(corr_pre, corr_post)

print("Average correlation summary:")
display(corr_summary.round(4))

print("Block correlation summary:")
display(block_corr_summary.round(4))

In [ ]:
print("Pre-COVID correlation matrix:")
display(corr_pre.round(4))

print("Post-COVID correlation matrix:")
display(corr_post.round(4))

print("Correlation difference matrix:")
display(corr_diff.round(4))


## 4.1 Statistical tests for changes in co-movement

This section tests whether key correlations changed statistically, not only visually. For individual pairs, we use the Fisher z-test for equality of correlations. For asset-class blocks, we use bootstrap confidence intervals for the change in average block correlation.


In [ ]:
key_pairs = [
    ("S&P500", "US IG Bonds"),
    ("S&P500", "US HY Bonds"),
    ("S&P500", "Gold"),
    ("S&P500", "Oil futures"),
    ("Eurostoxx 50", "S&P500"),
    ("US IG Bonds", "US HY Bonds"),
    ("Gold", "US IG Bonds"),
]

fisher_corr_tests = fisher_correlation_test(pre_covid, post_covid, key_pairs)
display(fisher_corr_tests.round(4))


In [ ]:
equity_assets = ["S&P500", "Eurostoxx 50", "Hang Seng", "MSCI EM", "SMI"]
bond_assets = ["US T 10-year Yield", "German Gov 10-year yield"]
credit_assets = ["US IG Bonds", "US HY Bonds"]
commodity_assets = ["Gold", "Oil futures"]

block_pairs_for_tests = [
    ("Equity-equity", equity_assets, equity_assets),
    ("Equity-bond", equity_assets, bond_assets),
    ("Equity-credit", equity_assets, credit_assets),
    ("Equity-commodity", equity_assets, commodity_assets),
    ("Bond-credit", bond_assets, credit_assets),
]

block_corr_tests = bootstrap_block_correlation_tests(
    pre_covid,
    post_covid,
    block_pairs_for_tests,
    n_bootstrap=N_BOOTSTRAP,
)

display(block_corr_tests.round(4))


In [ ]:
block_changes = block_corr_summary["Change"].sort_values(ascending=False)

print("Largest increases in block correlations:")
display(block_changes.head(5).round(4))

print("Largest decreases in block correlations:")
display(block_changes.tail(5).round(4))

### Figures: correlation changes

The first figure shows changes in asset-class block correlations. The second figure shows the full correlation difference matrix, where each cell is post-COVID correlation minus pre-COVID correlation.

In [ ]:
plot_block_correlation_change(block_corr_summary)
plot_correlation_difference_heatmap(corr_diff)


### Interpretation

The correlation results do not show a generalized increase in cross-asset correlations after COVID-19. The average pairwise correlation slightly declined after COVID.

However, the block-level results show important changes in the internal structure of co-movements. Credit markets became much more internally correlated, and the correlation between equities and credit increased strongly. This suggests that credit risk became more closely connected to broader risk-asset dynamics after COVID.

Government yield changes also became more synchronized, suggesting a stronger global rates or inflation channel. At the same time, equity-yield correlations declined, and commodity co-movement weakened.

Therefore, COVID-19 did not simply make all assets move together. Instead, it changed the correlation structure selectively, especially through credit and rates channels. This is important for portfolio management because diversification benefits changed across asset classes.

## 4.2 Global correlation-matrix change test

Pairwise correlations only show local changes. To test whether the overall co-movement structure changed, we compute the Frobenius norm of the difference between the post-COVID and pre-COVID correlation matrices.

This is preferred to a raw covariance-matrix distance in the main analysis because the dataset combines log returns, yield changes in basis points and a transformed oil series. Correlations are unit-free, so the global test is not mechanically dominated by variables measured in larger units.


In [ ]:
correlation_matrix_distance = correlation_frobenius_test(
    pre_covid,
    post_covid,
    n_bootstrap=N_BOOTSTRAP,
)

display(correlation_matrix_distance.round(6))


## 5. Risk factors: Principal Component Analysis

This section studies whether financial market risk became more concentrated in a smaller number of common factors after COVID-19.

Principal Component Analysis (PCA) transforms the cross-section of asset returns into orthogonal components. If the first principal components explain more variance after COVID, it means that a larger share of market risk is driven by common shocks.

To avoid high-volatility assets mechanically dominating the PCA, the analysis is performed on standardized returns.

In [ ]:
explained_pre, explained_post, pca_comparison, loadings_pre, loadings_post = (
    compare_pre_post_pca(pre_covid, post_covid)
)

print("PCA explained variance comparison:")
display(pca_comparison.head(10).round(4))

In [ ]:
pc12_pre = pca_comparison.loc[["PC1", "PC2"], "Explained_pre"].sum()
pc12_post = pca_comparison.loc[["PC1", "PC2"], "Explained_post"].sum()
pc12_change = pc12_post - pc12_pre

print(f"PC1 + PC2 explained variance pre-COVID:  {pc12_pre:.4f}")
print(f"PC1 + PC2 explained variance post-COVID: {pc12_post:.4f}")
print(f"Change: {pc12_change:.4f}")

In [ ]:
print("PC1 loadings pre-COVID:")
display(loadings_pre["PC1"].sort_values(ascending=False).round(4))

print("PC1 loadings post-COVID:")
display(loadings_post["PC1"].sort_values(ascending=False).round(4))

In [ ]:
print("PCA explained variance before COVID:")
display(explained_pre.round(4))

print("PCA explained variance after COVID:")
display(explained_post.round(4))

print("PCA loadings before COVID:")
display(loadings_pre.round(4))

print("PCA loadings after COVID:")
display(loadings_post.round(4))


### Bootstrap inference for PCA concentration

The PCA comparison is strengthened with bootstrap confidence intervals for the increase in common-factor concentration. If PC1 or PC1+PC2 increases significantly, risk became more concentrated in common shocks and diversification may have weakened.


In [ ]:
pca_bootstrap_tests = bootstrap_pca_concentration(
    pre_covid,
    post_covid,
    n_bootstrap=N_BOOTSTRAP,
)

display(pca_bootstrap_tests.round(4))


### Figure: PCA explained variance

The following figure compares the variance explained by the first principal components before and after COVID-19.

In [ ]:
plot_pca_explained_variance(pca_comparison)


### Interpretation

The PCA results show that risk became more concentrated after COVID-19.

The first principal component explains a larger share of total standardized variance after COVID. More importantly, the first two principal components together explain about 50% of total variance post-COVID, compared with about 43% before COVID.

This suggests that post-COVID market risk was more driven by common cross-asset factors. The loadings also indicate that the post-COVID first component is broader, involving equities, credit, FX, yields and gold. This supports the idea that risk shifted from a mainly equity-driven global risk factor toward a more integrated macro-financial risk factor.

This is one of the strongest pieces of evidence that the structure of risk changed after COVID-19.

## 6. Risk regimes: rolling volatility and rolling correlations

The previous sections compared pre-COVID and post-COVID averages. This section adds a dynamic view by computing rolling indicators.

We use a 252-day rolling window, which approximately corresponds to one trading year.

The objective is to observe whether COVID-19 coincides with a visible change in market risk regimes, especially in volatility and cross-asset correlation.

In [ ]:
rolling_vol = compute_rolling_volatility(
    returns_df,
    window=252,
)

rolling_corr = compute_rolling_average_correlation(
    returns_df,
    window=252,
)

print("Rolling volatility shape:", rolling_vol.shape)
print("Rolling correlation length:", len(rolling_corr))

rolling_vol.tail()

In [ ]:
print("Rolling annualized volatility, last observations:")
display(rolling_vol.tail().round(4))

print("Rolling average correlation, last observations:")
display(rolling_corr.tail().round(4))


### Figure: rolling annualized volatility

The following figure shows rolling annualized volatility for selected assets. The vertical dashed line marks 11 March 2020.

In [ ]:
selected_assets = [
    "S&P500",
    "US IG Bonds",
    "US HY Bonds",
    "Oil futures",
    "Gold",
]

plot_rolling_volatility(rolling_vol, selected_assets)


### Figure: rolling average cross-asset correlation

The following figure shows the rolling average pairwise correlation across all assets. This helps assess whether cross-asset diversification changed dynamically around COVID-19.

In [ ]:
plot_rolling_average_correlation(rolling_corr)


### Interpretation

The rolling volatility figure shows that COVID-19 coincided with a sharp increase in risk for several assets, especially oil, equities and credit-related instruments. This confirms that the pandemic period was associated with a clear risk-regime shock.

The rolling average correlation figure provides a dynamic view of diversification. If the average correlation rises during stress periods, diversification becomes less effective. If it later declines, the shock may be more temporary. This helps distinguish between a short-lived crisis episode and a more persistent change in the risk structure.

Overall, the rolling evidence supports the idea that COVID-19 was a major risk-regime event. Combined with the pre/post tables, it suggests that the risk structure changed selectively rather than uniformly.

## 6.1 Structural-break analysis: equity-bond diversification regime

The rolling figures are descriptive. To add a formal regime/break dimension from the course, we focus on the 252-day rolling correlation between the S&P 500 and US investment-grade bonds. This is economically important because equity-bond diversification is central to portfolio construction.

We estimate a Bai-Perron-style segmented mean-break model by dynamic programming. The objective is to identify persistent shifts in the average rolling correlation. Higher or positive equity-bond correlation means weaker diversification; lower or negative correlation means stronger diversification.


In [ ]:
rolling_spx_ig_corr = compute_rolling_pair_correlation(
    returns_df,
    "S&P500",
    "US IG Bonds",
    window=252,
)

spx_ig_break_segments = segmented_mean_breaks(
    rolling_spx_ig_corr,
    max_breaks=3,
    min_segment=126,
)

print("Rolling S&P500 vs US IG Bonds correlation, last observations:")
display(rolling_spx_ig_corr.tail().round(4))

print("Detected correlation-regime segments:")
display(spx_ig_break_segments)


In [ ]:
plt.figure(figsize=(12, 5))
rolling_spx_ig_corr.plot()
for break_date in spx_ig_break_segments["Start"].iloc[1:]:
    plt.axvline(break_date, linestyle="--", linewidth=1)
plt.axhline(0, linewidth=1)
plt.title("252-day rolling correlation: S&P 500 vs US IG Bonds")
plt.ylabel("Rolling correlation")
plt.tight_layout()
plt.show()


## 6.2 Portfolio implications: did diversification change?

The final empirical question is whether the change in risk structure matters for a portfolio manager. We compare an equal-weight portfolio, a long-only minimum-variance portfolio and an equal-risk-contribution portfolio.

For this section, the analysis is restricted to return-like investable series only. Government yield changes are excluded because they are measured in basis points and are not directly investable returns. Oil is excluded from the baseline portfolio exercise because its post-COVID transformation is dominated by the April 2020 negative-price episode. Oil and yield changes remain included in the broader risk-structure, PCA, correlation and robustness analyses.

The first table asks: if an investor kept the same pre-COVID allocation, did realized portfolio risk and diversification change under the post-COVID covariance matrix? The second table asks: if the investor recalibrated the allocation separately before and after COVID, how would optimal weights change?


In [ ]:
pre_portfolio = pre_covid[INVESTABLE_RETURN_ASSETS].copy()
post_portfolio = post_covid[INVESTABLE_RETURN_ASSETS].copy()

portfolio_summary, portfolio_weights, portfolio_risk_contributions = portfolio_implication_tables(
    pre_portfolio,
    post_portfolio,
)

print("Portfolio analysis using investable return-like assets only:")
display(portfolio_summary.round(4))

print("Pre/post optimal weights:")
display(portfolio_weights.round(4))

print("Portfolio risk contributions:")
display(portfolio_risk_contributions.round(4))


In [ ]:
minvar_erc_weights = portfolio_weights.set_index("Asset")[[
    "Min-var weight pre",
    "Min-var weight post",
    "ERC weight pre",
    "ERC weight post",
]]

minvar_erc_weights.plot(kind="bar", figsize=(12, 6))
plt.title("Portfolio weights before and after COVID")
plt.ylabel("Weight")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


## 7. Robustness checks

The baseline analysis uses 11 March 2020 as the COVID breakpoint. This date corresponds to the point at which COVID-19 was officially characterized as a pandemic.

However, financial markets started reacting before and after that date. Therefore, we test whether the main conclusions are robust to alternative breakpoints.

We also run a robustness check excluding oil futures, because WTI oil prices became negative in April 2020 and therefore require a special transformation.

In [ ]:
# Robustness functions are defined directly in this notebook.
# No external Python robustness module is required.
print("Self-contained robustness functions available.")


In [ ]:
breakpoints = [
    "2020-02-24",
    "2020-03-11",
    "2020-03-23",
    "2020-06-01",
]

breakpoint_robustness = run_breakpoint_robustness_clean(
    returns_df,
    breakpoints=breakpoints,
)

print("Breakpoint robustness table:")
display(breakpoint_robustness.round(4))


In [ ]:
no_oil_results = run_no_oil_robustness(
    returns_df,
    breakpoint="2020-03-11",
)

print(no_oil_results.keys())

In [ ]:
no_oil_pca = no_oil_results["pca_comparison_no_oil"]
no_oil_corr = no_oil_results["correlation_summary_no_oil"]
no_oil_risk = no_oil_results["risk_comparison_no_oil"]

print("No-oil correlation summary:")
display(no_oil_corr.round(4))

print("No-oil PCA comparison:")
display(no_oil_pca.head(5).round(4))

print("No-oil risk comparison:")
display(no_oil_risk.round(4))

In [ ]:
print("No-oil robustness summary:")

print("Risk comparison without oil:")
display(no_oil_results["risk_comparison_no_oil"].round(4))

print("Average correlation without oil:")
display(no_oil_results["correlation_summary_no_oil"].round(4))

print("First five PCA components without oil:")
display(no_oil_results["pca_comparison_no_oil"].head(5).round(4))

print("PCA loadings are computed and stored in no_oil_results, but not displayed in full to keep the notebook concise.")

### Additional robustness: excluding yields and using weekly frequency

The no-yields robustness checks whether PCA and co-movement results are driven by yield changes measured in basis points. The weekly robustness reduces daily noise and checks whether conclusions survive at a lower frequency.


In [ ]:
no_yields_results = run_no_yields_robustness(
    returns_df,
    breakpoint="2020-03-11",
)

weekly_results = run_weekly_robustness(
    returns_df,
    breakpoint="2020-03-11",
)

print("No-yields robustness summary:")
print("Average correlation without yield-change series:")
display(no_yields_results["correlation_summary_no_yields"].round(4))
print("First five PCA components without yield-change series:")
display(no_yields_results["pca_comparison_no_yields"].head(5).round(4))
print("Risk comparison without yield-change series:")
display(no_yields_results["risk_comparison_no_yields"].round(4))

print("Weekly-frequency robustness summary:")
print("Average correlation at weekly frequency:")
display(weekly_results["correlation_summary_weekly"].round(4))
print("First five PCA components at weekly frequency:")
display(weekly_results["pca_comparison_weekly"].head(5).round(4))
print("Risk comparison at weekly frequency:")
display(weekly_results["risk_comparison_weekly"].round(4))

### Interpretation

The breakpoint robustness check tests whether the main conclusion depends on choosing 11 March 2020 as the COVID split date. If the direction of the main results remains similar across alternative breakpoints, the conclusion is more credible.

The volatility part of the breakpoint table is separated into return-like assets, yield changes and oil. This avoids averaging volatility changes across incompatible units.

The no-oil robustness check is important because oil futures experienced an exceptional dislocation in April 2020. If the main PCA and correlation conclusions remain visible after excluding oil, then the results are not driven only by the oil shock.

Overall, the robustness checks are used to confirm whether the project’s conclusion is stable: COVID-19 changed the structure of risk partially and selectively, rather than uniformly across all assets.


## 8. Final conclusion

The empirical evidence supports a **partial change** in the structure of risk after COVID-19.

The results do not support the simple idea that all assets became riskier or that all markets became more correlated. Average cross-asset correlation did not increase after COVID, and several assets experienced lower or stable volatility.

However, the structure of risk changed in more specific and economically meaningful ways.

First, risk levels increased strongly in selected areas, especially oil, government yield changes, credit bonds, gold and the S&P500. This suggests that the COVID shock affected multiple risk channels: energy risk, rates risk, credit risk and equity risk.

Second, the correlation structure changed selectively. Credit markets became more internally correlated, government yield changes became more synchronized, and the equity-credit relationship strengthened. This implies that diversification benefits changed, especially for portfolios combining equities and credit.

Third, PCA results suggest stronger common-factor concentration after COVID. The first two principal components explain a larger share of total standardized variance after COVID than before COVID. This result is robust across alternative COVID breakpoints and remains visible even when oil futures are excluded.

Therefore, the answer to the research question is:

**Yes, but only partially. Since COVID-19, the structure of risk in financial markets has changed selectively rather than uniformly. The main change is not a generalized rise in all correlations, but stronger common-factor concentration after COVID, especially through credit, rates and cross-asset channels.**


In [ ]:
print("Notebook completed successfully.")
print("All tables and figures were displayed inline in this notebook.")
print("No result files were written by this notebook.")
